# Part A: removal experiments on pretrained models (Colab)

This notebook runs every Part A experiment of the write-up (`docs/post/README.md` §4.3) on one machine:

1. `01_ioi_dataset.py`: the 600 IOI prompts and their ABC twins, logged as the artifact `ioi-prompts`.
2. `02_gpt2_removals.py`: GPT-2 small, hypotheses H1–H3. The name movers removed as a set, all 144 single-head removals, keystones, attenuation, and the charts `charts/interaction_matrix`, `charts/name_mover_removal`, `charts/keystones`.
3. `03_models.py`: H4. The top three heads removed in GPT-2 small and medium and Pythia-160M, -410M, -1.4B; one run per model plus the summary run `models-H4` with the chart `charts/compensation_by_model`.
4. `04_checkpoints.py`: H5. The same measurement at eleven Pythia-410M training checkpoints; the summary run `pythia-410m-H5` charts compensation against training step.

**Before running:** Runtime → Change runtime type → **A100 GPU**. Add the `WANDB_API_KEY` secret. Total time is about 30–45 minutes, most of it downloading checkpoints. Every script skips work whose output already exists in `outputs/`, so after a disconnect re-run from cell 2.

GPT-2 small and the four smaller models were also run on the local GTX 1070 (runs tagged `local-gtx1070`); the runs here are the reference set, and the agreement between the two is reported in the write-up.

## 1. Check the GPU
Expect an A100. If this prints nothing, the runtime type is still CPU.

In [ ]:
!nvidia-smi -L

## 2. Get the code
Clones the repo. While the repo is private, add a Colab secret `GITHUB_TOKEN` (a fine-grained GitHub token with read access to `kvenanzi/trophic-cascade`); once it is public no token is needed. `BRANCH` lets this run from a feature branch before it is merged; re-running the cell pulls the latest commit.

In [ ]:
import os, subprocess

from google.colab import userdata

REPO = "kvenanzi/trophic-cascade"
BRANCH = "main"
try:
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN}@github.com/{REPO}.git" if TOKEN else f"https://github.com/{REPO}.git"
if not os.path.isdir("trophic-cascade"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL], check=True)
else:
    subprocess.run(["git", "-C", "trophic-cascade", "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", "trophic-cascade", "pull", "-q"], check=True)
!git -C trophic-cascade log --oneline -1

## 3. Make the package importable and install what Colab lacks
The clone goes on `sys.path` (for this notebook) and on `PYTHONPATH` (for the scripts run with `!python`), so `import trophic` reads the code straight from the clone. The pip line adds only what Colab does not ship. TransformerLens is capped below 4.0, which removed `HookedTransformer`.

In [ ]:
import sys
ROOT = os.path.abspath("trophic-cascade")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.environ["PYTHONPATH"] = ROOT
%pip install -q "transformer-lens>=2.16,<4.0" wandb wandb-workspaces
import torch, transformers, transformer_lens, trophic
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| transformers", transformers.__version__,
      "| transformer_lens", getattr(transformer_lens, "__version__", "?"), "| trophic from", os.path.dirname(trophic.__file__))

## 4. Log in to Weights & Biases
The API key comes from Colab Secrets (key icon on the left, `WANDB_API_KEY`, *Notebook access* on). Every run below logs to `within-noise/trophic-cascade`.

In [ ]:
import wandb
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()
%cd {ROOT}

## 5. The prompt set

In [ ]:
!python scripts/01_ioi_dataset.py

## 6. GPT-2 small: H1–H3
About 3 minutes. Prints the baseline (mean logit difference about 3.6, IO preferred on 99% of prompts), the name-mover removal (H1), the keystones (H2), and the attenuation test (H3).

In [ ]:
!python scripts/02_gpt2_removals.py

## 7. Five models: H4
Pythia-1.4B is loaded in float16 to save host memory and cast to float32 on the GPU, so every model is measured at the same precision.

In [ ]:
!python scripts/03_models.py --batch-size 32

## 8. Pythia-410M checkpoints: H5
Eleven checkpoints from step 1,000 to 143,000 (the final model). The top three heads are re-selected at each checkpoint.

In [ ]:
!python scripts/04_checkpoints.py

## 9. What to look at
- Project page, **Runs** table: group by `job_type`. The `removals` run holds H1–H3; `models-*` and `pythia-410m-step*` hold H4 and H5.
- Run `gpt2-small-removals` → **Media**: the interaction heatmap, the before/after bar chart of the name-mover removal, and the keystone scatter.
- Run `models-H4`: compensation by model with 95% intervals. Run `pythia-410m-H5`: compensation against checkpoint step.
- **Artifacts**: `gpt2-removals`, `models-*`, `checkpoints-pythia-410m` hold every number the write-up quotes.